In [1]:
import os
import glob
import pandas as pd
import numpy as np
from pathlib import Path

# ===============================
# CONFIG
# ===============================
BASE_PATH = Path("../dataset")
CLEAN_PATH = Path("../cleaned")
LOG_PATH = Path("../cleaned/cleaning_log.csv")
CHUNK_LIMIT_MB = 200  # files larger than this will use chunk processing
CHUNK_SIZE = 200000

CLEAN_PATH.mkdir(exist_ok=True, parents=True)

print("Starting full dataset cleaning...")

# ===============================
# HELPER FUNCTIONS
# ===============================

def standardize_columns(df):
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_", regex=False)
        .str.replace("-", "_", regex=False)
        .str.replace("(", "", regex=False)
        .str.replace(")", "", regex=False)
    )
    return df

def convert_dates(df):
    for col in df.columns:
        if "date" in col or "time" in col:
            try:
                df[col] = pd.to_datetime(df[col], errors="coerce")
            except:
                pass
    return df

def convert_numeric(df):
    for col in df.columns:
        if df[col].dtype == "object":
            try:
                df[col] = pd.to_numeric(
                    df[col].astype(str).str.replace(",", ""), 
                    errors="ignore"
                )
            except:
                pass
    return df

def basic_clean(df):
    initial_rows = len(df)
    initial_cols = len(df.columns)

    df = standardize_columns(df)
    df = convert_dates(df)
    df = convert_numeric(df)

    df = df.replace(["NA", "N/A", "", " "], np.nan)
    df = df.drop_duplicates()
    df = df.dropna(how="all")

    final_rows = len(df)

    return df, {
        "initial_rows": initial_rows,
        "final_rows": final_rows,
        "initial_cols": initial_cols,
        "final_cols": len(df.columns),
        "duplicates_removed": initial_rows - final_rows
    }

def save_cleaned(df, original_path):
    save_path = CLEAN_PATH / original_path.relative_to(BASE_PATH)
    save_path.parent.mkdir(parents=True, exist_ok=True)

    if original_path.suffix == ".csv":
        df.to_csv(save_path, index=False)
    elif original_path.suffix == ".xlsx":
        df.to_excel(save_path, index=False)

    return save_path

def clean_small_file(filepath):
    if filepath.suffix == ".csv":
        df = pd.read_csv(filepath, low_memory=False)
    elif filepath.suffix == ".xlsx":
        df = pd.read_excel(filepath)
    else:
        return None

    df_clean, stats = basic_clean(df)
    save_path = save_cleaned(df_clean, filepath)

    stats["file"] = str(filepath)
    stats["saved_to"] = str(save_path)
    return stats

def clean_large_csv(filepath):
    chunks = []
    total_stats = None

    for chunk in pd.read_csv(filepath, chunksize=CHUNK_SIZE):
        chunk_clean, stats = basic_clean(chunk)
        chunks.append(chunk_clean)

        if total_stats is None:
            total_stats = stats
        else:
            total_stats["initial_rows"] += stats["initial_rows"]
            total_stats["final_rows"] += stats["final_rows"]
            total_stats["duplicates_removed"] += stats["duplicates_removed"]

    df = pd.concat(chunks, ignore_index=True)

    save_path = save_cleaned(df, filepath)

    total_stats["file"] = str(filepath)
    total_stats["saved_to"] = str(save_path)

    return total_stats

# ===============================
# MAIN EXECUTION
# ===============================

logs = []

all_files = list(BASE_PATH.rglob("*"))

for file in all_files:
    if file.suffix in [".csv", ".xlsx"]:
        print(f"\nProcessing: {file}")

        file_size_mb = file.stat().st_size / (1024 * 1024)

        try:
            if file.suffix == ".csv" and file_size_mb > CHUNK_LIMIT_MB:
                stats = clean_large_csv(file)
            else:
                stats = clean_small_file(file)

            if stats:
                logs.append(stats)
                print(f"Completed: {file.name}")

        except Exception as e:
            print(f"Error in {file}: {e}")

# Save cleaning log
log_df = pd.DataFrame(logs)
log_df.to_csv(LOG_PATH, index=False)

print("\nAll files cleaned successfully.")
print(f"Cleaning log saved at: {LOG_PATH}")

Starting full dataset cleaning...

Processing: ..\dataset\CEA Data\CO2_DatabaseVersion_19_2022_23.xlsx


C:\Users\Ohi\AppData\Local\Temp\ipykernel_16712\2733531717.py:49: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(
C:\Users\Ohi\AppData\Local\Temp\ipykernel_16712\2733531717.py:49: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(
C:\Users\Ohi\AppData\Local\Temp\ipykernel_16712\2733531717.py:49: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(
C:\Users\Ohi\AppData\Local\Temp\ipykernel_16712\2733531717.py:49: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] =

Completed: CO2_DatabaseVersion_19_2022_23.xlsx

Processing: ..\dataset\CEA Data\CO2_DatabaseVersion_19_2022_23_preview.csv
Completed: CO2_DatabaseVersion_19_2022_23_preview.csv

Processing: ..\dataset\CEA Data\CO2_DatabaseVersion_19_2022_23_preview_preview.csv
Completed: CO2_DatabaseVersion_19_2022_23_preview_preview.csv

Processing: ..\dataset\CEA Data\CO2_DatabaseVersion_19_2022_23_preview_preview_preview.csv
Completed: CO2_DatabaseVersion_19_2022_23_preview_preview_preview.csv

Processing: ..\dataset\CEA Data\CO2_Database_Version_20.0_2023_24.xlsx


C:\Users\Ohi\AppData\Local\Temp\ipykernel_16712\2733531717.py:49: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(
C:\Users\Ohi\AppData\Local\Temp\ipykernel_16712\2733531717.py:49: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(
C:\Users\Ohi\AppData\Local\Temp\ipykernel_16712\2733531717.py:49: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(
C:\Users\Ohi\AppData\Local\Temp\ipykernel_16712\2733531717.py:49: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] =

Completed: CO2_Database_Version_20.0_2023_24.xlsx

Processing: ..\dataset\CEA Data\CO2_Database_Version_20.0_2023_24_preview.csv
Completed: CO2_Database_Version_20.0_2023_24_preview.csv

Processing: ..\dataset\CEA Data\CO2_Database_Version_20.0_2023_24_preview_preview.csv
Completed: CO2_Database_Version_20.0_2023_24_preview_preview.csv

Processing: ..\dataset\CEA Data\CO2_Database_Version_20.0_2023_24_preview_preview_preview.csv
Completed: CO2_Database_Version_20.0_2023_24_preview_preview_preview.csv

Processing: ..\dataset\CEA Data\CO2_Database_V_21.0 (1).xlsx


...

Completed: State-wise monthly wind speed.xlsx

Processing: ..\dataset\IDP\wheather\temperature.csv


C:\Users\Ohi\AppData\Local\Temp\ipykernel_16712\2733531717.py:49: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(
C:\Users\Ohi\AppData\Local\Temp\ipykernel_16712\2733531717.py:49: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(
C:\Users\Ohi\AppData\Local\Temp\ipykernel_16712\2733531717.py:49: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(
C:\Users\Ohi\AppData\Local\Temp\ipykernel_16712\2733531717.py:49: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] =

Completed: temperature.csv

Processing: ..\dataset\IDP\wheather\wind_speed_tel_hr_punjab_sw_pb_1970_2025.csv


C:\Users\Ohi\AppData\Local\Temp\ipykernel_16712\2733531717.py:40: UserWarning: Parsing dates in %d-%m-%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df[col] = pd.to_datetime(df[col], errors="coerce")
C:\Users\Ohi\AppData\Local\Temp\ipykernel_16712\2733531717.py:49: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(
C:\Users\Ohi\AppData\Local\Temp\ipykernel_16712\2733531717.py:49: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(
C:\Users\Ohi\AppData\Local\Temp\ipykernel_16712\2733531717.py:49: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exception

Completed: wind_speed_tel_hr_punjab_sw_pb_1970_2025.csv

All files cleaned successfully.
Cleaning log saved at: ..\cleaned\cleaning_log.csv


In [2]:
from pathlib import Path

CLEAN_PATH = Path("../cleaned")

for file in CLEAN_PATH.rglob("*preview*.csv"):
    print("Deleting:", file)
    file.unlink()

print("Preview files removed.")

Deleting: ..\cleaned\CEA Data\CO2_DatabaseVersion_19_2022_23_preview.csv
Deleting: ..\cleaned\CEA Data\CO2_DatabaseVersion_19_2022_23_preview_preview.csv
Deleting: ..\cleaned\CEA Data\CO2_DatabaseVersion_19_2022_23_preview_preview_preview.csv
Deleting: ..\cleaned\CEA Data\CO2_Database_Version_20.0_2023_24_preview.csv
Deleting: ..\cleaned\CEA Data\CO2_Database_Version_20.0_2023_24_preview_preview.csv
Deleting: ..\cleaned\CEA Data\CO2_Database_Version_20.0_2023_24_preview_preview_preview.csv
Deleting: ..\cleaned\CEA Data\CO2_Database_V_21.0 (1)_preview.csv
Deleting: ..\cleaned\CEA Data\CO2_Database_V_21.0 (1)_preview_preview.csv
Deleting: ..\cleaned\CEA Data\CO2_Database_V_21.0 (1)_preview_preview_preview.csv
Deleting: ..\cleaned\CEA Data\installed-capacity_preview.csv
Deleting: ..\cleaned\CEA Data\installed-capacity_preview_preview.csv
Deleting: ..\cleaned\CEA Data\installed-capacity_preview_preview_preview.csv
Deleting: ..\cleaned\CEA Data\installed-capacity_punjab_2022-2025_preview.cs

In [3]:
import pandas as pd

df = pd.read_csv("../cleaned/IDP/daily-power-generation.csv", nrows=5)
df.head()

,id,date,region,state_name,state_code,sector,station_type,power_station,power_station_unit,monitored_capacity,todays_gen_prgm,todays_gen_act
0,0,2017-09-01,Northern,Delhi,7.0,State,Thermal,Rajghat Tps,Unit 1,67.5,0.00,0.00
1,1,2017-09-01,Northern,Delhi,7.0,State,Thermal,Rajghat Tps,Unit 2,67.5,0.00,0.00
2,2,2017-09-01,Northern,Delhi,7.0,State,Ther (Gt),I.P.Ccpp,Single Unit,270.0,1.80,0.87
3,3,2017-09-01,Northern,Delhi,7.0,State,Ther (Gt),Pragati Ccgt-Iii,Single Unit,1500.0,6.00,10.89
4,4,2017-09-01,Northern,Delhi,7.0,State,Ther (Gt),Pragati Ccpp,Single Unit,330.4,5.49,6.53


In [4]:
import pandas as pd
from pathlib import Path

CLEAN_PATH = Path("../cleaned")
LOG_PATH = Path("../logs")
LOG_PATH.mkdir(exist_ok=True)

log_entries = []

# Load generation data
gen_path = CLEAN_PATH / "IDP/daily-power-generation.csv"
df = pd.read_csv(gen_path, parse_dates=["date"])

initial_rows = len(df)

# Standardize station_type
df["station_type"] = df["station_type"].str.lower().str.strip()

# Aggregate national total
daily_total = df.groupby("date")["todays_gen_act"].sum().reset_index()
daily_total.rename(columns={"todays_gen_act": "total_generation"}, inplace=True)

# Aggregate by station type
daily_type = df.pivot_table(
    index="date",
    columns="station_type",
    values="todays_gen_act",
    aggfunc="sum"
).reset_index()

# Merge totals + types
daily_gen = pd.merge(daily_total, daily_type, on="date", how="left")

# Save
master_path = CLEAN_PATH / "master_generation_daily.csv"
daily_gen.to_csv(master_path, index=False)

# Logging
log_entries.append({
    "stage": "generation_aggregation",
    "initial_rows": initial_rows,
    "final_rows": len(daily_gen),
    "columns_created": len(daily_gen.columns),
    "file_saved": str(master_path)
})

pd.DataFrame(log_entries).to_csv(LOG_PATH / "model_build_log.csv", index=False)

print("Generation master created.")
daily_gen.head()

Generation master created.


,date,total_generation,hydro,nuclear,ther (dg),ther (gt),thermal
0,2017-09-01,3154.78,497.68,89.18,0.27,115.16,2416.96
1,2017-09-02,3186.81,495.63,92.28,0.27,110.56,2452.54
2,2017-09-03,3134.00,473.05,92.57,0.27,107.62,2425.34
3,2017-09-04,3341.64,500.36,92.62,0.27,120.71,2593.30
4,2017-09-05,3377.16,470.16,92.71,0.27,121.06,2658.32


In [5]:
import pandas as pd
import numpy as np
from pathlib import Path

CLEAN_PATH = Path("../cleaned")
LOG_PATH = Path("../logs")
LOG_PATH.mkdir(exist_ok=True)

audit_summary = []
preview_data = {}

def get_middle_rows(df, n=2):
    if len(df) < n:
        return df
    mid = len(df) // 2
    return df.iloc[mid:mid+n]

for file in CLEAN_PATH.rglob("*"):
    if file.suffix in [".csv", ".xlsx"]:
        try:
            if file.suffix == ".csv":
                df = pd.read_csv(file, low_memory=False)
            else:
                df = pd.read_excel(file)

            rows, cols = df.shape
            col_list = list(df.columns)
            dtypes = dict(df.dtypes.astype(str))
            missing_percent = (df.isna().mean() * 100).to_dict()

            middle_rows = get_middle_rows(df, 2)
            preview_data[str(file)] = middle_rows

            audit_summary.append({
                "file": str(file),
                "rows": rows,
                "cols": cols,
                "columns": ", ".join(col_list),
                "dtypes": str(dtypes),
                "avg_missing_percent": np.mean(list(missing_percent.values()))
            })

            print(f"Audited: {file}")

        except Exception as e:
            print(f"Error auditing {file}: {e}")

# Save summary CSV
summary_df = pd.DataFrame(audit_summary)
summary_df.to_csv(LOG_PATH / "data_audit_summary.csv", index=False)

# Save previews to Excel (each file in separate sheet)
with pd.ExcelWriter(LOG_PATH / "data_audit_preview.xlsx") as writer:
    for name, df_preview in preview_data.items():
        sheet_name = name.split("/")[-1][:30]
        df_preview.to_excel(writer, sheet_name=sheet_name, index=False)

print("Data audit completed.")

Audited: ..\cleaned\cleaning_log.csv
Audited: ..\cleaned\master_generation_daily.csv
Audited: ..\cleaned\CEA Data\CO2_DatabaseVersion_19_2022_23.xlsx
Audited: ..\cleaned\CEA Data\CO2_Database_Version_20.0_2023_24.xlsx
Audited: ..\cleaned\CEA Data\CO2_Database_V_21.0 (1).xlsx
Audited: ..\cleaned\CEA Data\installed-capacity.csv
Audited: ..\cleaned\CEA Data\installed-capacity_punjab_2022-2025.csv
Audited: ..\cleaned\CEA Data\installed-capacity_punjab_2022-2025_sector wise.csv
Audited: ..\cleaned\Daily power genration\Daily_Power_Gen_Source_march_23.csv
Audited: ..\cleaned\Daily power genration\Daily_Power_Gen_States_march_23.csv
Audited: ..\cleaned\IDP\daily-coal-stocks.csv
Audited: ..\cleaned\IDP\daily-power-generation.csv
Audited: ..\cleaned\IDP\daily-power-outage.csv
Audited: ..\cleaned\IDP\daily-renewable-energy-generation.csv
Audited: ..\cleaned\IDP\interstate-generating-stationisgs-renewable-energy-generation.csv
Audited: ..\cleaned\IDP\state-control-renewable-energy-generation.csv


ValueError: Invalid character \ found in sheet title

In [6]:
import pandas as pd
import numpy as np
from pathlib import Path

# ===============================
# PATHS
# ===============================
CLEAN_PATH = Path("../cleaned")
LOG_PATH = Path("../logs")
LOG_PATH.mkdir(exist_ok=True)

# ===============================
# FILES TO INSPECT (CORE ONLY)
# ===============================
FILES = {
    "generation": CLEAN_PATH / "IDP/daily-power-generation.csv",
    "outage": CLEAN_PATH / "IDP/daily-power-outage.csv",
    "coal_stock": CLEAN_PATH / "IDP/daily-coal-stocks.csv",
    "renewable": CLEAN_PATH / "IDP/daily-renewable-energy-generation.csv",
    "co2": CLEAN_PATH / "CEA Data/CO2_Database_V_21.0 (1).xlsx",
    "price_sample": CLEAN_PATH / "prices of energy/2018/jan.csv",
    "temperature": CLEAN_PATH / "IDP/wheather/temperature.csv",
    "rainfall": CLEAN_PATH / "IDP/wheather/daily-rainfall-at-state-level.csv",
}

# ===============================
# HELPERS
# ===============================
def get_middle_rows(df, n=2):
    if len(df) < n:
        return df
    mid = len(df) // 2
    return df.iloc[mid:mid+n]

schema_summary = []
preview_data = {}

# ===============================
# INSPECTION LOOP
# ===============================
for name, file in FILES.items():
    print(f"\n==============================")
    print(f"Inspecting: {name}")
    print(f"Path: {file}")
    print("==============================")

    try:
        if file.suffix == ".csv":
            df = pd.read_csv(file, low_memory=False)
        else:
            df = pd.read_excel(file)

        rows, cols = df.shape
        print(f"Rows: {rows}, Columns: {cols}")

        print("\nColumn Names:")
        print(df.columns.tolist())

        print("\nDtypes:")
        print(df.dtypes)

        missing_percent = (df.isna().mean() * 100).round(2)
        print("\nMissing %:")
        print(missing_percent)

        middle_rows = get_middle_rows(df, 2)
        print("\n2 Middle Rows:")
        print(middle_rows)

        preview_data[name] = middle_rows

        schema_summary.append({
            "dataset": name,
            "file": str(file),
            "rows": rows,
            "cols": cols,
            "columns": ", ".join(df.columns),
            "avg_missing_percent": float(np.mean(missing_percent))
        })

        # Extra: Special check for generation
        if name == "generation":
            if "station_type" in df.columns:
                print("\nUnique station_type values:")
                print(df["station_type"].unique())

    except Exception as e:
        print(f"Error reading {file}: {e}")

# ===============================
# SAVE REPORTS
# ===============================
summary_df = pd.DataFrame(schema_summary)
summary_df.to_csv(LOG_PATH / "core_schema_summary.csv", index=False)

with pd.ExcelWriter(LOG_PATH / "core_schema_preview.xlsx") as writer:
    for name, df_preview in preview_data.items():
        df_preview.to_excel(writer, sheet_name=name[:30], index=False)

print("\nSchema inspection complete.")
print("Saved:")
print(" - core_schema_summary.csv")
print(" - core_schema_preview.xlsx")


Inspecting: generation
Path: ..\cleaned\IDP\daily-power-generation.csv
Rows: 2583818, Columns: 12

Column Names:
['id', 'date', 'region', 'state_name', 'state_code', 'sector', 'station_type', 'power_station', 'power_station_unit', 'monitored_capacity', 'todays_gen_prgm', 'todays_gen_act']

Dtypes:
id                      int64
date                   object
region                 object
state_name             object
state_code            float64
sector                 object
station_type           object
power_station          object
power_station_unit     object
monitored_capacity    float64
todays_gen_prgm       float64
todays_gen_act        float64
dtype: object

Missing %:
id                    0.00
date                  0.00
region                0.00
state_name            0.00
state_code            0.49
sector                0.49
station_type          0.49
power_station         0.00
power_station_unit    0.00
monitored_capacity    0.60
todays_gen_prgm       0.00
todays_gen_act   

In [7]:
import pandas as pd
import numpy as np
import os
import re
from pathlib import Path

# ===============================
# PATHS
# ===============================
BASE_PATH = Path("../cleaned")
OUTPUT_PATH = Path("../cleaned_v2")
LOG_PATH = Path("../logs")

OUTPUT_PATH.mkdir(exist_ok=True)
LOG_PATH.mkdir(exist_ok=True)

log_records = []

# ===============================
# HELPERS
# ===============================

def clean_columns(df):
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(r"\s+", "_", regex=True)
        .str.replace(r"[^\w_]", "", regex=True)
    )
    return df

def convert_objects_to_numeric(df):
    for col in df.columns:
        if df[col].dtype == "object":
            df[col] = df[col].astype(str).str.replace(",", "")
            df[col] = df[col].str.replace(r"[^\d\.\-]", "", regex=True)
            df[col] = pd.to_numeric(df[col], errors="ignore")
    return df

def standardize_dates(df):
    for col in df.columns:
        if "date" in col or "delivery" in col or "dates" in col:
            try:
                df[col] = pd.to_datetime(df[col], errors="ignore")
            except:
                pass
    return df

def drop_empty(df):
    if df.shape[0] == 0:
        return None
    return df

def process_file(file_path):
    try:
        if file_path.suffix == ".csv":
            df = pd.read_csv(file_path, low_memory=False)
        else:
            df = pd.read_excel(file_path)

        initial_rows = df.shape[0]
        initial_cols = df.shape[1]

        df = clean_columns(df)
        df = standardize_dates(df)
        df = convert_objects_to_numeric(df)
        df = drop_empty(df)

        if df is None:
            print(f"Skipped empty file: {file_path}")
            return

        final_rows = df.shape[0]
        final_cols = df.shape[1]

        # Save
        relative = file_path.relative_to(BASE_PATH)
        save_path = OUTPUT_PATH / relative
        save_path.parent.mkdir(parents=True, exist_ok=True)

        if file_path.suffix == ".csv":
            df.to_csv(save_path, index=False)
        else:
            df.to_excel(save_path, index=False)

        log_records.append({
            "file": str(file_path),
            "initial_rows": initial_rows,
            "final_rows": final_rows,
            "initial_cols": initial_cols,
            "final_cols": final_cols
        })

        print(f"Processed: {file_path}")

    except Exception as e:
        print(f"Error processing {file_path}: {e}")

# ===============================
# RUN OVER ALL FILES
# ===============================
for root, dirs, files in os.walk(BASE_PATH):
    for file in files:
        if file.endswith((".csv", ".xlsx")):
            process_file(Path(root) / file)

# ===============================
# SAVE CLEANING LOG
# ===============================
log_df = pd.DataFrame(log_records)
log_df.to_csv(LOG_PATH / "full_cleaning_v2_log.csv", index=False)

print("\nALL FILES STANDARDIZED AND SAVED TO cleaned_v2")

C:\Users\Ohi\AppData\Local\Temp\ipykernel_16712\3128441460.py:38: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors="ignore")
C:\Users\Ohi\AppData\Local\Temp\ipykernel_16712\3128441460.py:38: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors="ignore")
C:\Users\Ohi\AppData\Local\Temp\ipykernel_16712\3128441460.py:45: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_datetime(df[col], errors="ignore")


Processed: ..\cleaned\cleaning_log.csv
Processed: ..\cleaned\master_generation_daily.csv


C:\Users\Ohi\AppData\Local\Temp\ipykernel_16712\3128441460.py:38: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors="ignore")
C:\Users\Ohi\AppData\Local\Temp\ipykernel_16712\3128441460.py:38: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors="ignore")
C:\Users\Ohi\AppData\Local\Temp\ipykernel_16712\3128441460.py:38: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors="ignore")
C:\Users\Ohi\AppData\Local\Temp\ipykernel_16712\3128441460.py:38: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric w

Processed: ..\cleaned\CEA Data\CO2_DatabaseVersion_19_2022_23.xlsx


C:\Users\Ohi\AppData\Local\Temp\ipykernel_16712\3128441460.py:38: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors="ignore")
C:\Users\Ohi\AppData\Local\Temp\ipykernel_16712\3128441460.py:38: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors="ignore")
C:\Users\Ohi\AppData\Local\Temp\ipykernel_16712\3128441460.py:38: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors="ignore")
C:\Users\Ohi\AppData\Local\Temp\ipykernel_16712\3128441460.py:38: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric w

...

C:\Users\Ohi\AppData\Local\Temp\ipykernel_16712\3128441460.py:45: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_datetime(df[col], errors="ignore")
C:\Users\Ohi\AppData\Local\Temp\ipykernel_16712\3128441460.py:38: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors="ignore")
C:\Users\Ohi\AppData\Local\Temp\ipykernel_16712\3128441460.py:38: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors="ignore")
C:\Users\Ohi\AppData\Local\Temp\ipykernel_16712\3128441460.py:38: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric

Processed: ..\cleaned\prices of energy\2023\jul.csv
Processed: ..\cleaned\prices of energy\2023\june.csv
Processed: ..\cleaned\prices of energy\2023\mar.csv
Processed: ..\cleaned\prices of energy\2023\may.csv


C:\Users\Ohi\AppData\Local\Temp\ipykernel_16712\3128441460.py:45: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_datetime(df[col], errors="ignore")
C:\Users\Ohi\AppData\Local\Temp\ipykernel_16712\3128441460.py:38: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors="ignore")
C:\Users\Ohi\AppData\Local\Temp\ipykernel_16712\3128441460.py:38: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors="ignore")
C:\Users\Ohi\AppData\Local\Temp\ipykernel_16712\3128441460.py:38: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric

Processed: ..\cleaned\prices of energy\2023\nov.csv
Processed: ..\cleaned\prices of energy\2023\oct.csv
Processed: ..\cleaned\prices of energy\2023\sep.csv
Processed: ..\cleaned\prices of energy\2024\feb.csv
Processed: ..\cleaned\prices of energy\2024\jan.csv
Processed: ..\cleaned\prices of energy\2024\mar.csv

ALL FILES STANDARDIZED AND SAVED TO cleaned_v2


C:\Users\Ohi\AppData\Local\Temp\ipykernel_16712\3128441460.py:45: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_datetime(df[col], errors="ignore")
C:\Users\Ohi\AppData\Local\Temp\ipykernel_16712\3128441460.py:38: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors="ignore")
C:\Users\Ohi\AppData\Local\Temp\ipykernel_16712\3128441460.py:38: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors="ignore")
C:\Users\Ohi\AppData\Local\Temp\ipykernel_16712\3128441460.py:38: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric

In [8]:
import pandas as pd
import numpy as np
import os
from pathlib import Path
import random
import json

BASE_PATH = Path("../cleaned_v2")
OUTPUT_PATH = Path("../logs")
OUTPUT_PATH.mkdir(exist_ok=True)

audit_records = []

# ===============================
# Helper: Get 2 middle rows
# ===============================
def get_middle_rows(df):
    if len(df) == 0:
        return None
    if len(df) < 2:
        return df.to_dict(orient="records")
    mid = len(df) // 2
    return df.iloc[mid:mid+2].to_dict(orient="records")

# ===============================
# Identify one price file per year
# ===============================
price_files_selected = {}

for root, dirs, files in os.walk(BASE_PATH):
    if "prices of energy" in root:
        for file in files:
            if file.endswith(".csv"):
                year = Path(root).name
                if year not in price_files_selected:
                    price_files_selected[year] = Path(root) / file

# ===============================
# Main Inspection Loop
# ===============================
for root, dirs, files in os.walk(BASE_PATH):
    for file in files:
        file_path = Path(root) / file

        if not file.endswith((".csv", ".xlsx")):
            continue

        # Price special handling
        if "prices of energy" in str(file_path):
            year = file_path.parent.name
            if price_files_selected.get(year) != file_path:
                continue  # skip other price files

        try:
            if file.endswith(".csv"):
                df = pd.read_csv(file_path, low_memory=False)
            else:
                df = pd.read_excel(file_path)

            rows, cols = df.shape
            col_names = list(df.columns)
            dtypes = dict(df.dtypes.astype(str))
            missing_percent = round(df.isna().mean().mean() * 100, 2)

            middle_rows = get_middle_rows(df)

            audit_records.append({
                "file": str(file_path),
                "rows": rows,
                "cols": cols,
                "columns": col_names,
                "dtypes": dtypes,
                "avg_missing_percent": missing_percent,
                "two_middle_rows": json.dumps(middle_rows)
            })

            print(f"Audited: {file_path}")

        except Exception as e:
            print(f"Error reading {file_path}: {e}")

# ===============================
# Save Final Audit
# ===============================
audit_df = pd.DataFrame(audit_records)
audit_df.to_csv(OUTPUT_PATH / "dataset_full_audit_report.csv", index=False)

print("\nAUDIT COMPLETE.")
print("Saved: logs/dataset_full_audit_report.csv")

Audited: ..\cleaned_v2\cleaning_log.csv
Audited: ..\cleaned_v2\master_generation_daily.csv
Audited: ..\cleaned_v2\CEA Data\CO2_DatabaseVersion_19_2022_23.xlsx
Audited: ..\cleaned_v2\CEA Data\CO2_Database_Version_20.0_2023_24.xlsx
Error reading ..\cleaned_v2\CEA Data\CO2_Database_V_21.0 (1).xlsx: Object of type NaTType is not JSON serializable
Audited: ..\cleaned_v2\CEA Data\installed-capacity.csv
Audited: ..\cleaned_v2\CEA Data\installed-capacity_punjab_2022-2025.csv
Audited: ..\cleaned_v2\CEA Data\installed-capacity_punjab_2022-2025_sector wise.csv
Audited: ..\cleaned_v2\Daily power genration\Daily_Power_Gen_Source_march_23.csv
Audited: ..\cleaned_v2\Daily power genration\Daily_Power_Gen_States_march_23.csv
Audited: ..\cleaned_v2\IDP\daily-coal-stocks.csv
Audited: ..\cleaned_v2\IDP\daily-power-generation.csv
Audited: ..\cleaned_v2\IDP\daily-power-outage.csv
Audited: ..\cleaned_v2\IDP\daily-renewable-energy-generation.csv
Audited: ..\cleaned_v2\IDP\interstate-generating-stationisgs-ren

In [10]:
import pandas as pd
import numpy as np
from pathlib import Path

# ==========================
# CONFIG
# ==========================
BASE_PATH = Path("../cleaned_v2")
LOG_DIR = Path("logs")
LOG_DIR.mkdir(exist_ok=True)

pd.set_option('future.no_silent_downcasting', True)

# Optional state mapping (expand later if needed)
STATE_MAP = {
    1: "Jammu & Kashmir",
    2: "Himachal Pradesh",
    3: "Punjab",
    4: "Chandigarh",
    5: "Uttarakhand",
    6: "Haryana",
    7: "Delhi",
    8: "Rajasthan",
    9: "Uttar Pradesh",
    10: "Bihar",
    20: "Odisha",
    21: "Jharkhand",
    22: "Chhattisgarh",
    23: "Madhya Pradesh",
    24: "Gujarat",
    27: "Maharashtra",
    29: "Karnataka",
    33: "Tamil Nadu",
    34: "Puducherry"
}

log_records = []

# ==========================
# CLEANING FUNCTION
# ==========================
def clean_dataframe(df):

    df = df.copy()

    # ---- Remove unnamed columns
    df = df.loc[:, ~df.columns.str.contains("^unnamed", case=False)]

    # ---- Replace "-" with NaN safely
    df = df.replace("-", np.nan)

    # ---- Standardize dates
    for col in df.columns:
        col_lower = col.lower()
        if "date" in col_lower or "dt__comm" in col_lower or "delivery" in col_lower:
            df[col] = pd.to_datetime(df[col].astype(str).str[:10], errors="coerce")

    # ---- Fix price numeric columns
    price_keywords = ["mcp", "mwh", "bid", "volume"]
    for col in df.columns:
        if any(k in col.lower() for k in price_keywords):
            df[col] = (
                df[col]
                .astype(str)
                .str.replace(",", "", regex=False)
                .str.replace(r"[^\d\.\-]", "", regex=True)
            )
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # ---- Fix categorical columns
    categorical_keywords = ["state", "region", "sector", "station", "type"]
    for col in df.columns:
        if any(k in col.lower() for k in categorical_keywords):
            df[col] = df[col].astype(str)

    # ---- State mapping
    if "state_code" in df.columns and "state_name" in df.columns:
        df["state_name"] = (
            pd.to_numeric(df["state_code"], errors="coerce")
            .map(STATE_MAP)
            .fillna(df["state_name"])
        )

    # ---- Drop fully empty columns
    df = df.dropna(axis=1, how="all")

    # ---- Drop duplicates
    df = df.drop_duplicates()

    return df

# ==========================
# PROCESS ALL FILES
# ==========================
for file in BASE_PATH.rglob("*"):
    if file.suffix.lower() in [".csv", ".xlsx"]:

        try:
            if file.suffix.lower() == ".csv":
                df = pd.read_csv(file, low_memory=False)
            else:
                df = pd.read_excel(file)

            original_rows, original_cols = df.shape

            df_clean = clean_dataframe(df)

            # Save back (overwrite v2)
            if file.suffix.lower() == ".csv":
                df_clean.to_csv(file, index=False)
            else:
                df_clean.to_excel(file, index=False)

            log_records.append({
                "file": str(file),
                "rows_before": original_rows,
                "cols_before": original_cols,
                "rows_after": df_clean.shape[0],
                "cols_after": df_clean.shape[1]
            })

            print(f"Updated: {file}")

        except Exception as e:
            print(f"Error cleaning {file}: {e}")

# ==========================
# SAVE LOG
# ==========================
log_df = pd.DataFrame(log_records)
log_df.to_csv(LOG_DIR / "v2_upgrade_log.csv", index=False)

print("\nV2 UPGRADE COMPLETE.")
print("Log saved to logs/v2_upgrade_log.csv")

Updated: ..\cleaned_v2\cleaning_log.csv
Updated: ..\cleaned_v2\master_generation_daily.csv
Updated: ..\cleaned_v2\CEA Data\CO2_DatabaseVersion_19_2022_23.xlsx
Updated: ..\cleaned_v2\CEA Data\CO2_Database_Version_20.0_2023_24.xlsx
Updated: ..\cleaned_v2\CEA Data\CO2_Database_V_21.0 (1).xlsx
Updated: ..\cleaned_v2\CEA Data\installed-capacity.csv
Updated: ..\cleaned_v2\CEA Data\installed-capacity_punjab_2022-2025.csv
Updated: ..\cleaned_v2\CEA Data\installed-capacity_punjab_2022-2025_sector wise.csv
Updated: ..\cleaned_v2\Daily power genration\Daily_Power_Gen_Source_march_23.csv
Updated: ..\cleaned_v2\Daily power genration\Daily_Power_Gen_States_march_23.csv
Updated: ..\cleaned_v2\IDP\daily-coal-stocks.csv
Updated: ..\cleaned_v2\IDP\daily-power-generation.csv
Updated: ..\cleaned_v2\IDP\daily-power-outage.csv
Updated: ..\cleaned_v2\IDP\daily-renewable-energy-generation.csv
Updated: ..\cleaned_v2\IDP\interstate-generating-stationisgs-renewable-energy-generation.csv
Updated: ..\cleaned_v2\ID

In [11]:
import pandas as pd
import numpy as np
import os
from pathlib import Path
import random
import json

BASE_PATH = Path("../cleaned_v2")
OUTPUT_PATH = Path("../logs")
OUTPUT_PATH.mkdir(exist_ok=True)

audit_records = []

# ===============================
# Helper: Get 2 middle rows
# ===============================
def get_middle_rows(df):
    if len(df) == 0:
        return None
    if len(df) < 2:
        return df.to_dict(orient="records")
    mid = len(df) // 2
    return df.iloc[mid:mid+2].to_dict(orient="records")

# ===============================
# Identify one price file per year
# ===============================
price_files_selected = {}

for root, dirs, files in os.walk(BASE_PATH):
    if "prices of energy" in root:
        for file in files:
            if file.endswith(".csv"):
                year = Path(root).name
                if year not in price_files_selected:
                    price_files_selected[year] = Path(root) / file

# ===============================
# Main Inspection Loop
# ===============================
for root, dirs, files in os.walk(BASE_PATH):
    for file in files:
        file_path = Path(root) / file

        if not file.endswith((".csv", ".xlsx")):
            continue

        # Price special handling
        if "prices of energy" in str(file_path):
            year = file_path.parent.name
            if price_files_selected.get(year) != file_path:
                continue  # skip other price files

        try:
            if file.endswith(".csv"):
                df = pd.read_csv(file_path, low_memory=False)
            else:
                df = pd.read_excel(file_path)

            rows, cols = df.shape
            col_names = list(df.columns)
            dtypes = dict(df.dtypes.astype(str))
            missing_percent = round(df.isna().mean().mean() * 100, 2)

            middle_rows = get_middle_rows(df)

            audit_records.append({
                "file": str(file_path),
                "rows": rows,
                "cols": cols,
                "columns": col_names,
                "dtypes": dtypes,
                "avg_missing_percent": missing_percent,
                "two_middle_rows": json.dumps(middle_rows)
            })

            print(f"Audited: {file_path}")

        except Exception as e:
            print(f"Error reading {file_path}: {e}")

# ===============================
# Save Final Audit
# ===============================
audit_df = pd.DataFrame(audit_records)
audit_df.to_csv(OUTPUT_PATH / "dataset_full_audit_report.csv", index=False)

print("\nAUDIT COMPLETE.")
print("Saved: logs/dataset_full_audit_report.csv")

Audited: ..\cleaned_v2\cleaning_log.csv
Audited: ..\cleaned_v2\master_generation_daily.csv
Error reading ..\cleaned_v2\CEA Data\CO2_DatabaseVersion_19_2022_23.xlsx: Object of type Timestamp is not JSON serializable
Error reading ..\cleaned_v2\CEA Data\CO2_Database_Version_20.0_2023_24.xlsx: Object of type Timestamp is not JSON serializable
Error reading ..\cleaned_v2\CEA Data\CO2_Database_V_21.0 (1).xlsx: Object of type Timestamp is not JSON serializable
Audited: ..\cleaned_v2\CEA Data\installed-capacity.csv
Audited: ..\cleaned_v2\CEA Data\installed-capacity_punjab_2022-2025.csv
Audited: ..\cleaned_v2\CEA Data\installed-capacity_punjab_2022-2025_sector wise.csv
Audited: ..\cleaned_v2\Daily power genration\Daily_Power_Gen_Source_march_23.csv
Audited: ..\cleaned_v2\Daily power genration\Daily_Power_Gen_States_march_23.csv
Audited: ..\cleaned_v2\IDP\daily-coal-stocks.csv
Audited: ..\cleaned_v2\IDP\daily-power-generation.csv
Audited: ..\cleaned_v2\IDP\daily-power-outage.csv
Audited: ..\cle

In [12]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE = Path("../cleaned_v2")

# ================================
# 1️⃣ DAILY GENERATION (STATE)
# ================================
gen = pd.read_csv(BASE / "IDP/daily-power-generation.csv", low_memory=False)
gen['date'] = pd.to_datetime(gen['date'], errors='coerce')

gen_state = (
    gen.groupby(['date', 'state_name'])['todays_gen_act']
    .sum()
    .reset_index()
    .rename(columns={'todays_gen_act': 'total_generation_mwh'})
)

# ================================
# 2️⃣ RENEWABLE GENERATION
# ================================
ren = pd.read_csv(BASE / "IDP/daily-renewable-energy-generation.csv", low_memory=False)
ren['date'] = pd.to_datetime(ren['date'], errors='coerce')

ren_state = (
    ren.groupby(['date', 'state_name'])['total_renewable_energy']
    .sum()
    .reset_index()
    .rename(columns={'total_renewable_energy': 'renewable_generation_mwh'})
)

# ================================
# 3️⃣ COAL STOCK
# ================================
coal = pd.read_csv(BASE / "IDP/daily-coal-stocks.csv", low_memory=False)
coal['date'] = pd.to_datetime(coal['date'], errors='coerce')

coal_state = (
    coal.groupby(['date', 'state_name'])['stock_days']
    .mean()
    .reset_index()
    .rename(columns={'stock_days': 'avg_coal_stock_days'})
)

# ================================
# 4️⃣ OUTAGES
# ================================
outage = pd.read_csv(BASE / "IDP/daily-power-outage.csv", low_memory=False)
outage['date'] = pd.to_datetime(outage['date'], errors='coerce')

out_state = (
    outage.groupby(['date', 'state_name'])['cap_under_outage']
    .sum()
    .reset_index()
    .rename(columns={'cap_under_outage': 'total_outage_mw'})
)

# ================================
# 5️⃣ CONSUMPTION (POSOCO)
# ================================
cons = pd.read_csv(BASE / "posoco/Indias_Electricity_Consumption_Dataset.csv")
cons['dates'] = pd.to_datetime(cons['dates'], errors='coerce')

cons_long = cons.melt(id_vars=['dates'], var_name='state_name', value_name='consumption_mwh')
cons_long = cons_long.rename(columns={'dates': 'date'})

# ================================
# 6️⃣ MARKET PRICE (DAILY AVG)
# ================================
price_files = list((BASE / "prices of energy").rglob("*.csv"))

price_list = []

for f in price_files:
    try:
        df = pd.read_csv(f)
        if 'delivery' in df.columns:
            df['delivery'] = pd.to_datetime(df['delivery'], errors='coerce')
        elif 'time_block' in df.columns:
            df['delivery'] = pd.to_datetime(df['time_block'].str[:10], errors='coerce')
        else:
            continue

        if 'mcp_rsmwh' in df.columns:
            daily_price = (
                df.groupby('delivery')['mcp_rsmwh']
                .mean()
                .reset_index()
                .rename(columns={'delivery': 'date', 'mcp_rsmwh': 'avg_market_price'})
            )
            price_list.append(daily_price)
    except:
        continue

price_state = pd.concat(price_list, ignore_index=True)

# ================================
# 7️⃣ MERGE EVERYTHING
# ================================
master = gen_state.merge(ren_state, on=['date','state_name'], how='left')
master = master.merge(coal_state, on=['date','state_name'], how='left')
master = master.merge(out_state, on=['date','state_name'], how='left')
master = master.merge(cons_long, on=['date','state_name'], how='left')

# price is national (not state-level)
master = master.merge(price_state, on='date', how='left')

# ================================
# 8️⃣ CLEAN FINAL
# ================================
master = master.sort_values(['date','state_name'])
master = master.drop_duplicates()

master.to_csv("../cleaned_v2/master_state_daily_2015_2024.csv", index=False)

print("MASTER DATASET CREATED.")
print("Shape:", master.shape)

MASTER DATASET CREATED.
Shape: (47826, 8)


In [13]:
import pandas as pd
import numpy as np
from pathlib import Path

MASTER_PATH = Path("../cleaned_v2/master_state_daily_2015_2024.csv")
df = pd.read_csv(MASTER_PATH)

df['date'] = pd.to_datetime(df['date'], errors='coerce')

print("\n================ BASIC INFO ================")
print("Shape:", df.shape)
print("Date range:", df['date'].min(), "to", df['date'].max())
print("Total states:", df['state_name'].nunique())

print("\n================ DTYPES ================")
print(df.dtypes)

print("\n================ MISSING % ================")
print((df.isnull().mean()*100).sort_values(ascending=False))

print("\n================ SUMMARY STATS ================")
print(df.describe().T)

print("\n================ SKEWNESS ================")
print(df.skew(numeric_only=True))

print("\n================ KURTOSIS ================")
print(df.kurtosis(numeric_only=True))

print("\n================ NEGATIVE VALUE CHECK ================")
for col in df.select_dtypes(include=np.number).columns:
    print(col, "Negative Count:", (df[col] < 0).sum())

print("\n================ ZERO VALUE % ================")
for col in df.select_dtypes(include=np.number).columns:
    print(col, "Zero %:", (df[col] == 0).mean()*100)

print("\n================ CORRELATION MATRIX ================")
print(df.corr(numeric_only=True))

print("\n================ OUTLIER CHECK (IQR) ================")
for col in df.select_dtypes(include=np.number).columns:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    outliers = ((df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)).sum()
    print(col, "Outliers:", outliers)

print("\n================ FEATURE ENGINEERING CHECK ================")

df['renewable_ratio'] = df['renewable_generation_mwh'] / df['total_generation_mwh']
df['supply_demand_gap'] = df['total_generation_mwh'] - df['consumption_mwh']
df['outage_ratio'] = df['total_outage_mw'] / (df['total_generation_mwh'] + 1e-6)

print(df[['renewable_ratio','supply_demand_gap','outage_ratio']].describe())

print("\n================ PRICE VOLATILITY ================")
print("Daily Price Std:", df['avg_market_price'].std())

print("\n================ NATIONAL DAILY AGGREGATE ================")
national = df.groupby('date').agg({
    'total_generation_mwh':'sum',
    'renewable_generation_mwh':'sum',
    'consumption_mwh':'sum',
    'avg_market_price':'mean'
}).reset_index()

print(national.describe())

print("\n================ TOP 5 HIGH PRICE DAYS ================")
print(df.sort_values('avg_market_price', ascending=False).head())

print("\n================ TOP 5 LOW COAL STOCK DAYS ================")
print(df.sort_values('avg_coal_stock_days').head())

print("\n================ RANDOM 5 ROWS SAMPLE ================")
print(df.sample(5))


================ BASIC INFO ================
Shape: (47826, 8)
Date range: 2017-09-01 00:00:00 to 2025-04-22 00:00:00
Total states: 18

================ DTYPES ================
date                        datetime64[ns]
state_name                          object
total_generation_mwh               float64
renewable_generation_mwh           float64
avg_coal_stock_days                float64
total_outage_mw                    float64
consumption_mwh                    float64
avg_market_price                   float64
dtype: object

================ MISSING % ================
consumption_mwh             100.000000
avg_market_price             85.359428
avg_coal_stock_days          71.975495
renewable_generation_mwh     32.066240
total_generation_mwh          0.000000
state_name                    0.000000
date                          0.000000
total_outage_mw               0.000000
dtype: float64

================ SUMMARY STATS ================
                            count          

In [14]:
import pandas as pd
import numpy as np
from pathlib import Path

MASTER_PATH = Path("../cleaned_v2/master_state_daily_2015_2024.csv")
OUTPUT_PATH = Path("../cleaned_v2/master_dataset_diagnostics.txt")

df = pd.read_csv(MASTER_PATH)
df['date'] = pd.to_datetime(df['date'], errors='coerce')

df['renewable_ratio'] = df['renewable_generation_mwh'] / (df['total_generation_mwh'] + 1e-6)
df['supply_demand_gap'] = df['total_generation_mwh'] - df['consumption_mwh']
df['outage_ratio'] = df['total_outage_mw'] / (df['total_generation_mwh'] + 1e-6)

numeric_cols = df.select_dtypes(include=np.number).columns

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:

    f.write("====================================================\n")
    f.write("MASTER DATASET DIAGNOSTIC REPORT\n")
    f.write("====================================================\n\n")

    f.write("DATASET OVERVIEW\n")
    f.write(f"Total Rows: {len(df)}\n")
    f.write(f"Total Columns: {len(df.columns)}\n")
    f.write(f"Date Range: {df['date'].min()} to {df['date'].max()}\n")
    f.write(f"Total States: {df['state_name'].nunique()}\n\n")

    f.write("MISSING VALUE ANALYSIS (%):\n")
    missing = (df.isnull().mean()*100).sort_values(ascending=False)
    for col, val in missing.items():
        f.write(f"{col}: {val:.2f}%\n")
    f.write("\n")

    f.write("NUMERIC COLUMN STATISTICS\n")
    for col in numeric_cols:
        series = df[col]
        Q1 = series.quantile(0.25)
        Q3 = series.quantile(0.75)
        IQR = Q3 - Q1
        outliers = ((series < Q1 - 1.5*IQR) | (series > Q3 + 1.5*IQR)).sum()

        f.write(f"\nCOLUMN: {col}\n")
        f.write(f"Mean: {series.mean()}\n")
        f.write(f"Std: {series.std()}\n")
        f.write(f"Min: {series.min()}\n")
        f.write(f"Max: {series.max()}\n")
        f.write(f"Skewness: {series.skew()}\n")
        f.write(f"Kurtosis: {series.kurtosis()}\n")
        f.write(f"Zero %: {(series == 0).mean()*100:.2f}\n")
        f.write(f"Negative Count: {(series < 0).sum()}\n")
        f.write(f"Outlier Count (IQR): {outliers}\n")

    f.write("\n====================================================\n")
    f.write("CORRELATION MATRIX\n")
    f.write("====================================================\n\n")
    f.write(df.corr(numeric_only=True).to_string())
    f.write("\n\n")

    f.write("TOP 5 HIGHEST SKEWNESS COLUMNS\n")
    skew_rank = df[numeric_cols].skew().sort_values(ascending=False)
    f.write(skew_rank.head().to_string())
    f.write("\n\n")

    f.write("TOP 5 HIGHEST KURTOSIS COLUMNS\n")
    kurt_rank = df[numeric_cols].kurtosis().sort_values(ascending=False)
    f.write(kurt_rank.head().to_string())
    f.write("\n\n")

    f.write("PRICE VOLATILITY\n")
    f.write(f"Price Std Dev: {df['avg_market_price'].std()}\n")
    f.write(f"99th Percentile Price: {df['avg_market_price'].quantile(0.99)}\n\n")

    f.write("EXTREME EVENTS\n")
    f.write("Top 5 Highest Price Days:\n")
    f.write(df.sort_values('avg_market_price', ascending=False).head().to_string())
    f.write("\n\n")

    f.write("Lowest 5 Coal Stock Days:\n")
    f.write(df.sort_values('avg_coal_stock_days').head().to_string())
    f.write("\n\n")

    f.write("AGGREGATED NATIONAL DAILY STATS\n")
    national = df.groupby('date').agg({
        'total_generation_mwh':'sum',
        'renewable_generation_mwh':'sum',
        'consumption_mwh':'sum',
        'avg_market_price':'mean'
    })
    f.write(national.describe().to_string())
    f.write("\n\n")

    f.write("MODEL READINESS OBSERVATION\n")
    f.write("Check skewness, kurtosis, price volatility, renewable ratio variability, and supply-demand gap correlation.\n")
    f.write("If heavy tails detected, consider log transform or CVaR-based risk modeling.\n")

print("TEXT DIAGNOSTIC REPORT CREATED:")
print(OUTPUT_PATH)

TEXT DIAGNOSTIC REPORT CREATED:
..\cleaned_v2\master_dataset_diagnostics.txt


In [15]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE = Path("../cleaned_v2")

# ======================================================
# 1️⃣ LOAD GENERATION
# ======================================================
gen = pd.read_csv(BASE / "IDP/daily-power-generation.csv", low_memory=False)
gen['date'] = pd.to_datetime(gen['date'], errors='coerce')

gen_state = (
    gen.groupby(['date','state_name'])['todays_gen_act']
    .sum()
    .reset_index()
    .rename(columns={'todays_gen_act':'total_generation_mwh'})
)

# ======================================================
# 2️⃣ LOAD RENEWABLE
# ======================================================
ren = pd.read_csv(BASE / "IDP/daily-renewable-energy-generation.csv", low_memory=False)
ren['date'] = pd.to_datetime(ren['date'], errors='coerce')

ren_state = (
    ren.groupby(['date','state_name'])['total_renewable_energy']
    .sum()
    .reset_index()
    .rename(columns={'total_renewable_energy':'renewable_generation_mwh'})
)

# ======================================================
# 3️⃣ LOAD COAL STOCK
# ======================================================
coal = pd.read_csv(BASE / "IDP/daily-coal-stocks.csv", low_memory=False)
coal['date'] = pd.to_datetime(coal['date'], errors='coerce')

coal_state = (
    coal.groupby(['date','state_name'])['stock_days']
    .mean()
    .reset_index()
    .rename(columns={'stock_days':'avg_coal_stock_days'})
)

# ======================================================
# 4️⃣ LOAD OUTAGE
# ======================================================
outage = pd.read_csv(BASE / "IDP/daily-power-outage.csv", low_memory=False)
outage['date'] = pd.to_datetime(outage['date'], errors='coerce')

out_state = (
    outage.groupby(['date','state_name'])['cap_under_outage']
    .sum()
    .reset_index()
    .rename(columns={'cap_under_outage':'total_outage_mw'})
)

# ======================================================
# 5️⃣ LOAD POSOCO DEMAND (FIX STATE NAMES)
# ======================================================
cons = pd.read_csv(BASE / "posoco/Indias_Electricity_Consumption_Dataset.csv")
cons['dates'] = pd.to_datetime(cons['dates'], errors='coerce')

cons_long = cons.melt(id_vars=['dates'], var_name='state_raw', value_name='consumption_mwh')
cons_long = cons_long.rename(columns={'dates':'date'})

# normalize state names
cons_long['state_name'] = (
    cons_long['state_raw']
    .str.replace("_"," ")
    .str.title()
)

# manual fixes for abbreviations
STATE_FIX = {
    "Hp":"Himachal Pradesh",
    "Jk":"Jammu & Kashmir",
    "Dnh":"Dadra And Nagar Haveli",
    "Dd":"Daman And Diu",
    "Up":"Uttar Pradesh",
    "Mp":"Madhya Pradesh",
    "Pondy":"Puducherry"
}

cons_long['state_name'] = cons_long['state_name'].replace(STATE_FIX)

cons_state = cons_long[['date','state_name','consumption_mwh']]

# ======================================================
# 6️⃣ LOAD PRICE (DAILY NATIONAL AVG)
# ======================================================
price_files = list((BASE / "prices of energy").rglob("*.csv"))
price_list = []

for f in price_files:
    try:
        dfp = pd.read_csv(f)
        if 'delivery' in dfp.columns:
            dfp['date'] = pd.to_datetime(dfp['delivery'], errors='coerce')
        elif 'time_block' in dfp.columns:
            dfp['date'] = pd.to_datetime(dfp['time_block'].str[:10], errors='coerce')
        else:
            continue
        
        if 'mcp_rsmwh' in dfp.columns:
            daily_price = (
                dfp.groupby('date')['mcp_rsmwh']
                .mean()
                .reset_index()
                .rename(columns={'mcp_rsmwh':'avg_market_price'})
            )
            price_list.append(daily_price)
    except:
        continue

price_df = pd.concat(price_list, ignore_index=True)

# ======================================================
# 7️⃣ MERGE ALL
# ======================================================
master = gen_state.merge(ren_state, on=['date','state_name'], how='left')
master = master.merge(coal_state, on=['date','state_name'], how='left')
master = master.merge(out_state, on=['date','state_name'], how='left')
master = master.merge(cons_state, on=['date','state_name'], how='left')
master = master.merge(price_df, on='date', how='left')

# ======================================================
# 8️⃣ SAFE FEATURE ENGINEERING
# ======================================================
MIN_GEN_THRESHOLD = 5

master['renewable_ratio'] = np.where(
    master['total_generation_mwh'] > MIN_GEN_THRESHOLD,
    master['renewable_generation_mwh'] / master['total_generation_mwh'],
    np.nan
)

master['outage_ratio'] = np.where(
    master['total_generation_mwh'] > MIN_GEN_THRESHOLD,
    master['total_outage_mw'] / master['total_generation_mwh'],
    np.nan
)

master['supply_demand_gap'] = master['total_generation_mwh'] - master['consumption_mwh']

# remove impossible ratios
master.loc[master['renewable_ratio'] > 1, 'renewable_ratio'] = np.nan

# ======================================================
# 9️⃣ KEEP ONLY VALID MODELING ROWS
# ======================================================
master = master[
    master['total_generation_mwh'].notnull() &
    master['consumption_mwh'].notnull() &
    master['avg_market_price'].notnull()
]

master = master.sort_values(['date','state_name'])
master = master.drop_duplicates()

# ======================================================
# 🔟 SAVE CLEAN MODEL DATASET
# ======================================================
master.to_csv(BASE / "MASTER_MODEL_READY.csv", index=False)

print("REBUILD COMPLETE")
print("Final Shape:", master.shape)
print("Date Range:", master['date'].min(), "to", master['date'].max())
print("States:", master['state_name'].nunique())

REBUILD COMPLETE
Final Shape: (6191, 11)
Date Range: 2017-09-01 00:00:00 to 2024-09-02 00:00:00
States: 18


In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import skew, kurtosis
from statsmodels.tsa.stattools import adfuller
from pathlib import Path

BASE = Path("../cleaned_v2")
df = pd.read_csv(BASE / "MASTER_MODEL_READY.csv")
df['date'] = pd.to_datetime(df['date'])

REPORT_PATH = BASE / "MASTER_ADVANCED_DIAGNOSTIC_REPORT.txt"

with open(REPORT_PATH, "w") as f:

    f.write("====================================================\n")
    f.write("ADVANCED MASTER DATASET DIAGNOSTIC REPORT\n")
    f.write("====================================================\n\n")

    # ---------------------------------------------------
    # BASIC STRUCTURE
    # ---------------------------------------------------
    f.write("DATASET STRUCTURE\n")
    f.write(f"Rows: {df.shape[0]}\n")
    f.write(f"Columns: {df.shape[1]}\n")
    f.write(f"Date Range: {df['date'].min()} to {df['date'].max()}\n")
    f.write(f"States: {df['state_name'].nunique()}\n\n")

    # ---------------------------------------------------
    # MISSING %
    # ---------------------------------------------------
    f.write("MISSING VALUES (%)\n")
    missing = df.isnull().mean()*100
    f.write(str(missing.sort_values(ascending=False)))
    f.write("\n\n")

    # ---------------------------------------------------
    # NUMERIC ANALYSIS
    # ---------------------------------------------------
    numeric_cols = df.select_dtypes(include=np.number).columns

    f.write("NUMERIC COLUMN ANALYSIS\n\n")

    for col in numeric_cols:
        series = df[col].dropna()
        if len(series) < 10:
            continue
        
        f.write(f"COLUMN: {col}\n")
        f.write(f"Mean: {series.mean()}\n")
        f.write(f"Std: {series.std()}\n")
        f.write(f"Min: {series.min()}\n")
        f.write(f"Max: {series.max()}\n")
        f.write(f"Skewness: {skew(series)}\n")
        f.write(f"Kurtosis: {kurtosis(series)}\n")
        f.write(f"Zero %: {(series==0).mean()*100}\n")
        f.write(f"Outlier % (IQR): ")

        Q1 = series.quantile(0.25)
        Q3 = series.quantile(0.75)
        IQR = Q3 - Q1
        outliers = ((series < Q1-1.5*IQR) | (series > Q3+1.5*IQR)).mean()*100
        f.write(f"{outliers}\n")

        # ADF stationarity
        try:
            adf_result = adfuller(series.sample(min(10000,len(series))))
            f.write(f"ADF p-value: {adf_result[1]}\n")
        except:
            f.write("ADF failed\n")

        f.write("\n")

    # ---------------------------------------------------
    # CORRELATION
    # ---------------------------------------------------
    f.write("CORRELATION MATRIX\n")
    corr = df[numeric_cols].corr()
    f.write(str(corr))
    f.write("\n\n")

    # ---------------------------------------------------
    # PRICE VOLATILITY ANALYSIS
    # ---------------------------------------------------
    f.write("PRICE VOLATILITY ANALYSIS\n")

    if 'avg_market_price' in df.columns:
        price = df['avg_market_price'].dropna()
        f.write(f"Price Std Dev: {price.std()}\n")
        f.write(f"95th Percentile: {price.quantile(0.95)}\n")
        f.write(f"99th Percentile: {price.quantile(0.99)}\n")
        f.write(f"Heavy Tail Ratio (99/50): {price.quantile(0.99)/price.median()}\n\n")

    # ---------------------------------------------------
    # EXTREME EVENT FREQUENCY
    # ---------------------------------------------------
    f.write("EXTREME EVENTS\n")
    for col in numeric_cols:
        series = df[col].dropna()
        if len(series) < 10:
            continue
        extreme = (series > series.quantile(0.99)).mean()*100
        f.write(f"{col} Top 1% Frequency: {extreme}%\n")
    f.write("\n")

    # ---------------------------------------------------
    # ROLLING VOLATILITY
    # ---------------------------------------------------
    f.write("ROLLING 30-DAY VOLATILITY (Price)\n")
    if 'avg_market_price' in df.columns:
        daily = df.groupby('date')['avg_market_price'].mean()
        rolling_vol = daily.rolling(30).std().mean()
        f.write(f"Avg 30-day Volatility: {rolling_vol}\n\n")

    # ---------------------------------------------------
    # REGIME SHIFT CHECK
    # ---------------------------------------------------
    f.write("REGIME SHIFT (Variance Pre/Post 2021)\n")
    if 'avg_market_price' in df.columns:
        pre = df[df['date'] < '2021-01-01']['avg_market_price'].std()
        post = df[df['date'] >= '2021-01-01']['avg_market_price'].std()
        f.write(f"Pre-2021 Std: {pre}\n")
        f.write(f"Post-2021 Std: {post}\n\n")

    f.write("====================================================\n")
    f.write("END OF REPORT\n")
    f.write("====================================================\n")

print("ADVANCED DIAGNOSTIC REPORT GENERATED")

ADVANCED DIAGNOSTIC REPORT GENERATED


In [18]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE = Path("../cleaned_v2")
df = pd.read_csv(BASE / "MASTER_MODEL_READY.csv")
df['date'] = pd.to_datetime(df['date'])

df = df.sort_values(['state_name','date']).reset_index(drop=True)

# =====================================================
# 1️⃣ RENEWABLE IMPUTATION (SAFE WITH TRANSFORM)
# =====================================================
df['renewable_generation_mwh'] = (
    df.groupby('state_name')['renewable_generation_mwh']
    .transform(lambda x: x.ffill())
)

df['renewable_generation_mwh'] = df['renewable_generation_mwh'].fillna(0)

df['renewable_ratio'] = (
    df.groupby('state_name')['renewable_ratio']
    .transform(lambda x: x.ffill())
)

df['renewable_ratio'] = df['renewable_ratio'].fillna(0)

# =====================================================
# 2️⃣ COAL HANDLING
# =====================================================
df['coal_available_flag'] = df['avg_coal_stock_days'].notnull().astype(int)

df['avg_coal_stock_days'] = (
    df.groupby('state_name')['avg_coal_stock_days']
    .transform(lambda x: x.fillna(x.median()))
)

# =====================================================
# 3️⃣ HEAVY TAIL STABILIZATION
# =====================================================
df['log_price'] = np.log(df['avg_market_price'])
df['log_outage_ratio'] = np.log1p(df['outage_ratio'])

# =====================================================
# 4️⃣ ROLLING FEATURES (STATE-WISE SAFE)
# =====================================================
for col in ['avg_market_price','consumption_mwh',
            'renewable_ratio','outage_ratio']:
    
    df[f'{col}_roll_mean_7'] = (
        df.groupby('state_name')[col]
        .transform(lambda x: x.rolling(7,min_periods=3).mean())
    )
    
    df[f'{col}_roll_std_7'] = (
        df.groupby('state_name')[col]
        .transform(lambda x: x.rolling(7,min_periods=3).std())
    )

# =====================================================
# 5️⃣ DEMAND GROWTH
# =====================================================
df['demand_growth'] = (
    df.groupby('state_name')['consumption_mwh']
    .transform(lambda x: x.pct_change())
)

# =====================================================
# 6️⃣ PRICE MOMENTUM
# =====================================================
df['price_momentum_7'] = (
    df.groupby('state_name')['avg_market_price']
    .transform(lambda x: x.pct_change(7))
)

# =====================================================
# 7️⃣ REGIME INDICATOR
# =====================================================
df['post_2021_regime'] = (df['date'] >= "2021-01-01").astype(int)

# =====================================================
# 8️⃣ STATE ID
# =====================================================
df['state_id'] = df['state_name'].astype('category').cat.codes

# =====================================================
# 9️⃣ DROP ONLY EARLY ROLLING NaNs
# =====================================================
df = df.dropna().reset_index(drop=True)

# =====================================================
# SAVE
# =====================================================
df.to_csv(BASE / "MASTER_RL_READY.csv", index=False)

print("RL READY DATASET CREATED")
print("Final shape:", df.shape)
print("Columns:", df.shape[1])

RL READY DATASET CREATED
Final shape: (4588, 26)
Columns: 26


In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import skew, kurtosis
from pathlib import Path

BASE = Path("../cleaned_v2")
df = pd.read_csv(BASE / "MASTER_RL_READY.csv")
df['date'] = pd.to_datetime(df['date'])

REPORT = BASE / "FINAL_RL_DATA_INSPECTION.txt"

with open(REPORT, "w") as f:

    f.write("====================================================\n")
    f.write("FINAL RL DATASET INSPECTION REPORT\n")
    f.write("====================================================\n\n")

    # -------------------------------------------------
    # BASIC STRUCTURE
    # -------------------------------------------------
    f.write("STRUCTURE\n")
    f.write(f"Rows: {df.shape[0]}\n")
    f.write(f"Columns: {df.shape[1]}\n")
    f.write(f"Date Range: {df['date'].min()} to {df['date'].max()}\n")
    f.write(f"States: {df['state_name'].nunique()}\n\n")

    # -------------------------------------------------
    # MISSING CHECK
    # -------------------------------------------------
    f.write("MISSING VALUES (%)\n")
    f.write(str(df.isnull().mean()*100))
    f.write("\n\n")

    # -------------------------------------------------
    # NUMERIC STABILITY
    # -------------------------------------------------
    numeric_cols = df.select_dtypes(include=np.number).columns
    f.write("NUMERIC STABILITY CHECK\n\n")

    for col in numeric_cols:
        series = df[col]
        f.write(f"COLUMN: {col}\n")
        f.write(f"Mean: {series.mean()}\n")
        f.write(f"Std: {series.std()}\n")
        f.write(f"Min: {series.min()}\n")
        f.write(f"Max: {series.max()}\n")
        f.write(f"Skewness: {skew(series)}\n")
        f.write(f"Kurtosis: {kurtosis(series)}\n")
        f.write(f"Zero %: {(series==0).mean()*100}\n\n")

    # -------------------------------------------------
    # ROLLING FEATURE VALIDATION
    # -------------------------------------------------
    f.write("ROLLING FEATURE SANITY\n")
    roll_cols = [c for c in df.columns if "roll" in c]
    for col in roll_cols:
        f.write(f"{col} Std: {df[col].std()}\n")
    f.write("\n")

    # -------------------------------------------------
    # REGIME BALANCE
    # -------------------------------------------------
    f.write("REGIME DISTRIBUTION\n")
    f.write(str(df['post_2021_regime'].value_counts(normalize=True)))
    f.write("\n\n")

    # -------------------------------------------------
    # STATE BALANCE
    # -------------------------------------------------
    f.write("STATE OBSERVATION DISTRIBUTION\n")
    f.write(str(df['state_name'].value_counts().describe()))
    f.write("\n\n")

    # -------------------------------------------------
    # RISK TAIL CHECK (LOG VARIABLES)
    # -------------------------------------------------
    f.write("LOG VARIABLE HEAVY TAIL CHECK\n")
    for col in ['log_price','log_outage_ratio']:
        f.write(f"{col} Skewness: {skew(df[col])}\n")
        f.write(f"{col} Kurtosis: {kurtosis(df[col])}\n")
    f.write("\n")

    # -------------------------------------------------
    # MULTICOLLINEARITY (HIGH CORR > 0.9)
    # -------------------------------------------------
    f.write("HIGH CORRELATIONS (>0.9)\n")
    corr = df[numeric_cols].corr().abs()
    high_corr = [(c1,c2,corr.loc[c1,c2])
                 for c1 in corr.columns
                 for c2 in corr.columns
                 if c1!=c2 and corr.loc[c1,c2] > 0.9]

    f.write(str(high_corr[:20]))
    f.write("\n\n")

    f.write("====================================================\n")
    f.write("END OF REPORT\n")
    f.write("====================================================\n")

print("FINAL RL INSPECTION COMPLETE")

FINAL RL INSPECTION COMPLETE


In [20]:
import os
from pathlib import Path

# Project root (change if needed)
PROJECT_ROOT = Path("../project_root")

# Define folders
folders = [
    PROJECT_ROOT / "data",
    PROJECT_ROOT / "data/raw",
    PROJECT_ROOT / "data/processed",
    PROJECT_ROOT / "data/rl_ready",
    PROJECT_ROOT / "notebooks",
    PROJECT_ROOT / "results",
    PROJECT_ROOT / "results/logs",
    PROJECT_ROOT / "results/models",
    PROJECT_ROOT / "results/figures"
]

for folder in folders:
    folder.mkdir(parents=True, exist_ok=True)

print("✅ Project folders created successfully.")

✅ Project folders created successfully.


In [21]:
df["date"] = pd.to_datetime(df["date"])

train_df = df[df["date"] < "2021-01-01"].copy()
test_df  = df[df["date"] >= "2021-01-01"].copy()

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

Train shape: (2456, 26)
Test shape: (2132, 26)


In [22]:
RL_FOLDER = PROJECT_ROOT / "data/rl_ready"

train_path = RL_FOLDER / "RL_TRAIN_DATA.csv"
test_path  = RL_FOLDER / "RL_TEST_DATA.csv"

train_df.to_csv(train_path, index=False)
test_df.to_csv(test_path, index=False)

print("✅ RL train and test datasets saved.")
print("Location:", RL_FOLDER)

✅ RL train and test datasets saved.
Location: ..\project_root\data\rl_ready


In [23]:
import os

print("Files in rl_ready:")
print(os.listdir(RL_FOLDER))

Files in rl_ready:
['RL_TEST_DATA.csv', 'RL_TRAIN_DATA.csv']


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

# =====================================================
# PATHS (YOUR ACTUAL STRUCTURE)
# =====================================================

BASE = Path(r"C:\Users\Ohi\Desktop\m\cleaned_v2")
RL_OUTPUT = Path(r"C:\Users\Ohi\Desktop\m\project_root\data\rl_ready")

master_path = BASE / "MASTER_RL_READY.csv"

train_path = RL_OUTPUT / "RL_TRAIN_DATA.csv"
test_path  = RL_OUTPUT / "RL_TEST_DATA.csv"

report_path = BASE / "FINAL_RL_DATA_INSPECTION.txt"

# =====================================================
# LOAD DATA
# =====================================================

df = pd.read_csv(master_path)
df["date"] = pd.to_datetime(df["date"])

# =====================================================
# DROP MULTICOLLINEAR FEATURES
# =====================================================

drop_cols = [
    "renewable_generation_mwh",
    "consumption_mwh_roll_mean_7",
    "avg_market_price",
    "renewable_ratio_roll_mean_7"
]

df = df.drop(columns=[c for c in drop_cols if c in df.columns])

# =====================================================
# FINAL RL FEATURE SET
# =====================================================

rl_features = [
    "total_generation_mwh",
    "consumption_mwh",
    "renewable_ratio",
    "avg_coal_stock_days",
    "coal_available_flag",
    "supply_demand_gap",
    "log_outage_ratio",
    "outage_ratio_roll_mean_7",
    "outage_ratio_roll_std_7",
    "log_price",
    "avg_market_price_roll_std_7",
    "price_momentum_7",
    "demand_growth",
    "consumption_mwh_roll_std_7",
    "post_2021_regime",
    "state_id"
]

meta_cols = ["date", "state_name"]

df = df[meta_cols + rl_features]

# =====================================================
# STRUCTURAL TIME SPLIT (NO LEAKAGE)
# =====================================================

train_df = df[df["date"] < "2022-01-01"].copy()
test_df  = df[df["date"] >= "2022-01-01"].copy()

# =====================================================
# SCALE NUMERIC FEATURES (FIT ON TRAIN ONLY)
# =====================================================

scaler = StandardScaler()

numeric_cols = [
    c for c in rl_features
    if c not in ["coal_available_flag", "post_2021_regime", "state_id"]
]

train_df[numeric_cols] = scaler.fit_transform(train_df[numeric_cols])
test_df[numeric_cols]  = scaler.transform(test_df[numeric_cols])

# =====================================================
# SAVE UPDATED FILES
# =====================================================

train_df.to_csv(train_path, index=False)
test_df.to_csv(test_path, index=False)

# =====================================================
# UPDATE FINAL REPORT
# =====================================================

report_text = f"""
FINAL RL DATASET (FROZEN VERSION)
==================================

Train Rows: {len(train_df)}
Test Rows: {len(test_df)}
Features Used: {len(rl_features)}
Dropped Columns: {drop_cols}
Scaling: StandardScaler (fit on train only)
Split Date: 2022-01-01

This dataset is final and ready for:
1) Carbon Digital Twin
2) Risk-Constrained PPO
3) Rolling Window Validation

No further cleaning required.
"""

with open(report_path, "w") as f:
    f.write(report_text)

print("✔ RL dataset fully updated and frozen.")

✔ RL dataset fully updated and frozen.
